|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Writing the kernel<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: coalesce the block-table gather<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import torch
import cudalib

This is the gather from PagedAttention, with everything else removed.

You have a table of rows in memory, an index that names the row each output
needs, and a sum over each gathered row. There is no softmax, no head, and no
block size.

Stage 07 builds the real one. This is the part that decides the speed.

In [ ]:
### run this cell: the data, and the oracle

NUM_ROWS, ROW_LEN = 200_000, 128

table = torch.randn(NUM_ROWS, ROW_LEN, device='cuda')
index = torch.randperm(NUM_ROWS, device='cuda').to(torch.int32)  # a block table
row_sums = torch.empty(NUM_ROWS, device='cuda')

oracle = table[index.long()].sum(dim=1)
useful_bytes = NUM_ROWS * ROW_LEN * 4
print(f'{useful_bytes/1e6:.0f} MB of rows to gather')

# Exercise 1: the obvious mapping, and its cost

Give each output row one thread. Let the thread walk its row. Write the
kernel, check it against the oracle, and measure the fraction of the card's
bandwidth that you reach.

In [ ]:
NAIVE = r"""
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <torch/extension.h>

// One thread per row. It walks the whole row on its own.
__global__ void gather_naive(const float* __restrict__ table,
                             const int* __restrict__ index,
                             float* __restrict__ row_sums,
                             const int num_rows, const int row_len) {
  const int row = blockIdx.x * blockDim.x + threadIdx.x;
  if (row >= num_rows) return;
  const float* row_values = table + (long)index[row] * row_len;
  float total = 0.f;
  for (int element = 0; element < row_len; ++element) total += row_values[element];
  row_sums[row] = total;
}

void gather(torch::Tensor table, torch::Tensor index, torch::Tensor row_sums) {
  const int num_rows = index.numel(), row_len = table.size(1);
  const int threads = 256;
  gather_naive<<<(num_rows + threads - 1)/threads, threads, 0,
                 at::cuda::getCurrentCUDAStream()>>>(
      table.data_ptr<float>(), index.data_ptr<int>(), row_sums.data_ptr<float>(),
      num_rows, row_len);
  C10_CUDA_KERNEL_LAUNCH_CHECK();
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("gather", &gather); }
"""

naive = cudalib.build_source('cc_gather_naive', NAIVE)

In [ ]:
naive.gather(table, index, row_sums)
print('correct:', torch.allclose(row_sums, oracle, rtol=1e-3, atol=1e-2))

peak = cudalib.peak_bandwidth(fresh=True)
ms_naive = cudalib.bench_ms(lambda: naive.gather(table,index,row_sums), best_of=3)
gbs_naive = useful_bytes/(ms_naive*1e-3)/1e9

print(f'thread per row: {ms_naive:.3f} ms  {gbs_naive:.0f} GB/s  {100*gbs_naive/peak:.0f}% of peak')

# Exercise 2: predict the fix before you write it

Do not measure first. Use the sector arithmetic. How much faster can a
coalesced version be?

You will not find a single number, and that is the interesting part. Give a
bound on each side instead. State the worst case for the naive mapping, and
the best case.

Where the measurement lands between those bounds is a fact about the cache.
You learn that fact only if you write the bounds down first.

In [ ]:
floats_per_sector = 32 // 4

# At one instruction the 32 threads of the warp sit ROW_LEN floats apart.
# So they touch 32 different sectors. They use 1 float row_sums of every 8.
worst_case = min(ROW_LEN, floats_per_sector)

# But the thread returns at once for row_values[element+1]. That byte sits in the sector
# it just paid for. Walk the whole row and the kernel uses every fetched
# byte. So the kernel wastes REQUESTS, not BYTES. It makes 8 requests where
# 1 belongs, and the cache must cover the difference.
best_case = 1

print(f'stride between neighbouring threads: {ROW_LEN} floats')
print(f'speedup if nothing is cached:        {worst_case}x')
print(f'speedup if the cache catches it all: {best_case}x')
print(f'\nso: somewhere in 1x..{worst_case}x, and where it lands tells you\n'
      f'how much of the waste the cache absorbed.')

# Exercise 3: one warp per row

The arithmetic stays the same. The answer stays the same. Change only which
thread touches which byte. The 32 [lanes](../../GLOSSARY.md#lane) of a warp walk one row together. They
then combine their partial sums with a [shuffle](../../GLOSSARY.md#warp-shuffle).

In [ ]:
COALESCED = r"""
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <torch/extension.h>

// One WARP per row. The 32 lanes walk the row together, so at every step they
// read 32 consecutive floats: one transaction instead of 32.
__global__ void gather_coalesced(const float* __restrict__ table,
                                 const int* __restrict__ index,
                                 float* __restrict__ row_sums,
                                 const int num_rows, const int row_len) {
  const int warp = (blockIdx.x * blockDim.x + threadIdx.x) / 32;
  const int lane = threadIdx.x % 32;
  if (warp >= num_rows) return;

  const float* row_values = table + (long)index[warp] * row_len;

  // lane L takes elements L, L+32, L+64, ... so neighbouring lanes are
  // always on neighbouring addresses
  float total = 0.f;
  for (int element = lane; element < row_len; element += 32) total += row_values[element];

  // the row's total is now spread across 32 registers. Butterfly them
  // together. Every lane in the mask must reach the shuffle.
  for (int stride = 16; stride > 0; stride >>= 1) total += __shfl_xor_sync(0xffffffff, total, stride);

  if (lane == 0) row_sums[warp] = total;
}

void gather(torch::Tensor table, torch::Tensor index, torch::Tensor row_sums) {
  const int num_rows = index.numel(), row_len = table.size(1);
  const int threads = 256, warps_per_block = threads / 32;
  gather_coalesced<<<(num_rows + warps_per_block - 1)/warps_per_block, threads, 0,
                     at::cuda::getCurrentCUDAStream()>>>(
      table.data_ptr<float>(), index.data_ptr<int>(), row_sums.data_ptr<float>(),
      num_rows, row_len);
  C10_CUDA_KERNEL_LAUNCH_CHECK();
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) { m.def("gather", &gather); }
"""

fast = cudalib.build_source('cc_gather_fast', COALESCED)

In [ ]:
fast.gather(table, index, row_sums)
print('correct:', torch.allclose(row_sums, oracle, rtol=1e-3, atol=1e-2))

ms_fast = cudalib.bench_ms(lambda: fast.gather(table,index,row_sums), best_of=3)
gbs_fast = useful_bytes/(ms_fast*1e-3)/1e9

print(f'thread per row: {ms_naive:.3f} ms  {gbs_naive:6.0f} GB/s  {100*gbs_naive/peak:3.0f}% of peak')
print(f'warp per row:   {ms_fast:.3f} ms  {gbs_fast:6.0f} GB/s  {100*gbs_fast/peak:3.0f}% of peak')
print(f'\nmeasured speedup:  {ms_naive/ms_fast:.2f}x')
print(f'predicted range:   {best_case}x .. {worst_case}x')

### What to look at

The measured speedup lands **between** your bounds, not on either one. It is
near 3x, against a worst case of 8x and a best case of 1x.

That number answers a question you cannot reason your way to. The caches
absorbed most of the wasted bytes. They did not absorb the wasted requests.

A thread that walks its row in order returns at once for the sector that it
just paid for. So it discards almost nothing. What it cannot recover is eight
requests where one belongs, and eight times the latency to hide.

This is why the memory-transactions notebook found a clean 1.15 residual and
this notebook did not. There, each thread read one float and never came back,
so the sector model told the whole story. Here it tells half.

The coalesced version also reports **more than 100 percent of peak**. That is
not a bug either. `useful_bytes` counts what the algorithm needs, not what the
hardware moves. Some gathered rows were still in L2 when a later warp asked
for them.

A number above 100 percent tells you that the cache helped. That is worth
knowing. Do not hide it by a change to the denominator.

Now do this to the real kernel:

    ./vc guide 8b